# 🎓 Predicción de Entrega de Tareas Escolares mediante Modelos Supervisados
**Maestría en Inteligencia Artificial Aplicada a la Educación e Investigación**  
**Asignatura:** Fundamentos de IA y Aprendizaje Automático  
**Unidad 10 — Proyecto Final Integrador (Actualizado)**  
**Metodología:** CRISP-ML (Cross-Industry Standard Process for Machine Learning)  
**Modelos:** Regresión Logística (Clasificación) y Regresión Lineal (Predicción Continua / LPM)  
**Estudiante:** Liskeidy Rosario Pérez  
**Profesor:** Dr. Darwin Muñoz, Ph.D  

---

## 📈 Metodología CRISP-ML en este Proyecto
CRISP-ML es un proceso sistemático para guiar proyectos de Machine Learning. Consta de 6 fases secuenciales y retroalimentadas:
1. **Comprensión del Negocio (Business Understanding)**
2. **Comprensión de los Datos (Data Understanding)**
3. **Preparación de los Datos (Data Preparation)**
4. **Modelado (Modeling)**
5. **Evaluación (Evaluation)**
6. **Despliegue y Monitoreo (Deployment & Monitoring)**

---
## 🏢 Fase 1 — Comprensión del Negocio (Business Understanding)

### **Objetivo Académico y de Negocio**
El fracaso y rezago escolar representan un problema crítico en los sistemas educativos. La entrega de tareas es un indicador temprano y robusto del compromiso académico. Identificar a tiempo a los estudiantes con riesgo de no entregar sus deberes permite a los docentes e instituciones intervenir oportunamente (con tutorías, recordatorios dirigidos o apoyo socioemocional) antes de que el incumplimiento afecte sus calificaciones finales.

### **Objetivos del Modelo**
1. **Clasificación (Regresión Logística)**: Predecir si un estudiante entregará o no la tarea actual (`Entrego` = 1 o 0) y estimar su probabilidad a partir de su nivel de asistencia, historial de tareas entregadas y si recibió un recordatorio.
2. **Regresión Continua (Regresión Lineal)**: Estimar el número de tareas entregadas anteriormente (`Tareas_Anteriores_Entregadas`) en base a la asistencia y recordatorio, permitiendo identificar patrones continuos de cumplimiento.
3. **Análisis de Probabilidad Lineal (LPM)**: Modelar la probabilidad de entrega mediante regresión lineal convencional para comparar con la regresión logística.

### **Criterios de Éxito**
- El modelo de clasificación debe superar el **80% de exactitud (Accuracy)** en el conjunto de prueba.
- El modelo de regresión lineal debe mostrar una correlación positiva fuerte ($R^2 > 0.50$) entre asistencia e historial de cumplimiento escolar.
- Obtener explicabilidad sobre qué variables impactan más en el comportamiento estudiantil.

---
## 📊 Fase 2 — Comprensión de los Datos (Data Understanding)

En esta fase cargamos el conjunto de datos de 24 estudiantes simulados, analizamos su estructura, realizamos análisis estadístico descriptivo y llevamos a cabo el Análisis Exploratorio de Datos (EDA) de forma gráfica.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    ConfusionMatrixDisplay, mean_squared_error, mean_absolute_error,
    r2_score, roc_curve, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías importadas correctamente.')

In [ ]:
# ── Carga del Dataset de 24 estudiantes ───────────────────────────────────
data = {
    'Estudiante': [
        'Ana Perez', 'Juan Rodriguez', 'Maria Gomez', 'Pedro Martinez',
        'Laura Diaz', 'Carlos Santos', 'Sofia Ramirez', 'Luis Herrera',
        'Valentina Cruz', 'Miguel Lopez', 'Camila Torres', 'Andres Castillo',
        'Daniela Reyes', 'Jose Fernandez', 'Paula Morales', 'Diego Vargas',
        'Gabriela Nunez', 'Javier Medina', 'Elena Rosario', 'Adrian Pena',
        'Natalia Jimenez', 'Samuel Ortiz', 'Isabella Guerrero', 'Kevin Batista'
    ],
    'Asistencia': [
        92, 65, 95, 58, 88, 72, 97, 60, 90, 55, 85, 68,
        93, 62, 89, 70, 96, 52, 87, 66, 91, 59, 94, 74
    ],
    'Tareas_Anteriores_Entregadas': [
        9, 4, 10, 3, 8, 5, 10, 4, 9, 2, 7, 5,
        9, 4, 8, 6, 10, 2, 8, 5, 9, 3, 10, 6
    ],
    'Recordatorio_Enviado': [
        1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0,
        1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1
    ],
    'Entrego': [
        1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0,
        1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1
    ]
}

df = pd.DataFrame(data)

print('── Primeras filas del dataset ──')
print(df.head(10).to_string(index=False))
print(f'\n── Forma del dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\n── Distribución de la variable objetivo (Entrego):')
print(df['Entrego'].value_counts().rename({1: 'Sí entregó (1)', 0: 'No entregó (0)'}).to_string())
print(f'\n── Estadísticas descriptivas de las variables predictoras:')
print(df[['Asistencia','Tareas_Anteriores_Entregadas','Recordatorio_Enviado']].describe().round(2).to_string())

In [ ]:
# ── Gráficos del Análisis Exploratorio de Datos (EDA) ───────────────────────
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análisis Exploratorio de Datos (EDA) — Comportamiento Estudiantil', fontsize=16, fontweight='bold', color='#1A252C')

# 1. Histograma de Asistencia por Estado de Entrega
sns.histplot(data=df, x='Asistencia', hue='Entrego', kde=True, ax=axes[0, 0], 
             palette={0: '#E74C3C', 1: '#2ECC71'}, multiple='stack', alpha=0.8, edgecolor='w')
axes[0, 0].set_title('Distribución de Asistencia vs Entrega', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('% Asistencia')
axes[0, 0].set_ylabel('Cantidad de Estudiantes')
axes[0, 0].legend(['Sí entregó (1)', 'No entregó (0)'])

# 2. Histograma de Tareas Anteriores por Estado de Entrega
sns.histplot(data=df, x='Tareas_Anteriores_Entregadas', hue='Entrego', kde=False, ax=axes[0, 1], 
             palette={0: '#E74C3C', 1: '#2ECC71'}, multiple='dodge', alpha=0.8, discrete=True)
axes[0, 1].set_title('Tareas Anteriores Entregadas vs Entrega', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Tareas Anteriores')
axes[0, 1].set_ylabel('Cantidad de Estudiantes')
axes[0, 1].legend(['Sí entregó (1)', 'No entregó (0)'])

# 3. Relación Asistencia vs Tareas Anteriores (Gráfico de Dispersión)
sns.scatterplot(data=df, x='Asistencia', y='Tareas_Anteriores_Entregadas', hue='Entrego', 
                palette={0: '#E74C3C', 1: '#2ECC71'}, s=100, style='Recordatorio_Enviado', ax=axes[1, 0])
axes[1, 0].set_title('Asistencia vs Tareas Anteriores', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('% Asistencia')
axes[1, 0].set_ylabel('Tareas Anteriores')

# 4. Mapa de Calor de Correlaciones
corr_matrix = df[['Asistencia', 'Tareas_Anteriores_Entregadas', 'Recordatorio_Enviado', 'Entrego']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f', square=True, ax=axes[1, 1], 
            cbar_kws={'label': 'Coeficiente de Correlación'})
axes[1, 1].set_title('Matriz de Correlación de Pearson', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_avanzado.png', dpi=150, bbox_inches='tight')
plt.show()

**Hallazgos Clave del EDA:**
1. **Correlación Extrema**: Existe una correlación de Pearson perfecta ($1.00$) entre `Recordatorio_Enviado` y `Entrego` en este conjunto simulado. Esto sugiere que las intervenciones docentes (recordatorios) son cruciales, o bien, existe un sesgo de selección donde el docente envía recordatorios solo a estudiantes con alta asistencia.
2. **Asistencia y Cumplimiento**: Quienes entregan la tarea muestran una asistencia promedio del 90.4%, en contraste con el 62% de quienes no la entregan. Asimismo, registran una media de 8.9 tareas anteriores entregadas, comparado con solo 3.8 en el grupo de no entrega.

---
## ⚙️ Fase 3 — Preparación de los Datos (Data Preparation)

Los algoritmos de regresión logística y lineal son sensibles a la escala de las variables debido a sus cálculos de coeficientes y regularización. Escalaremos las variables numéricas usando `StandardScaler` (media 0, desviación estándar 1). Además, dividiremos el dataset en un 70% de datos para entrenamiento y un 30% para prueba, estratificando el target para conservar la proporción de clases.

In [ ]:
# Definición de variables de clasificación
features_cls = ['Asistencia', 'Tareas_Anteriores_Entregadas', 'Recordatorio_Enviado']
X_cls = df[features_cls]
y_cls = df['Entrego']

# División 70/30 estratificada
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.30, random_state=42, stratify=y_cls
)

# Escalado de variables para Regresión Logística
scaler = StandardScaler()
X_train_cls_scaled = scaler.fit_transform(X_train_cls)
X_test_cls_scaled = scaler.transform(X_test_cls)

print('── Formas de los Datos de Clasificación ──')
print(f'Entrenamiento: {X_train_cls_scaled.shape[0]} estudiantes')
print(f'Prueba: {X_test_cls_scaled.shape[0]} estudiantes')
print(f'Media de variables escaladas (Asistencia, Tareas, Recordatorio): {X_train_cls_scaled.mean(axis=0).round(2)}')
print(f'Desv. Estándar de variables escaladas: {X_train_cls_scaled.std(axis=0).round(2)}')

---
## 🤖 Fase 4 — Modelado (Modeling)

En esta fase entrenaremos tres modelos:
1. **Regresión Logística**: Clasifica si un alumno entrega la tarea. Estima la probabilidad de entrega a través de la función sigmoide: $P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \dots + \beta_k X_k)}}$
2. **Regresión Lineal**: Estima una variable continua, en este caso predecimos el historial de tareas anteriores entregadas (`Tareas_Anteriores_Entregadas`) en base a su asistencia y si recibió un recordatorio.
3. **Modelo Lineal de Probabilidad (LPM)**: Entrenamos una regresión lineal ordinaria (OLS) sobre la variable binaria `Entrego` para ilustrar las diferencias de frontera de decisión respecto a la Regresión Logística.

In [ ]:
# ── 1. Entrenamiento de Regresión Logística ─────────────────────────────────
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_cls_scaled, y_train_cls)

print('── Coeficientes de Regresión Logística ──')
for name, coef in zip(features_cls, log_model.coef_[0]):
    print(f'  {name}: {coef:.4f}')
print(f'  Intercepto (Sesgo): {log_model.intercept_[0]:.4f}')

In [ ]:
# ── 2. Entrenamiento de Regresión Lineal ─────────────────────────────────────
# Predictoras continuas: Asistencia y Recordatorio
features_reg = ['Asistencia', 'Recordatorio_Enviado']
X_reg = df[features_reg]
y_reg = df['Tareas_Anteriores_Entregadas']

# Separamos en train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.30, random_state=42
)

lin_model = LinearRegression()
lin_model.fit(X_train_reg, y_train_reg)

print('── Coeficientes de Regresión Lineal (Predicción de Tareas Anteriores) ──')
for name, coef in zip(features_reg, lin_model.coef_):
    print(f'  {name}: {coef:.4f}')
print(f'  Intercepto: {lin_model.intercept_:.4f}')

In [ ]:
# ── 3. Modelo de Probabilidad Lineal (LPM) ──────────────────────────────────
# Ajustamos regresión lineal a la variable y_cls binaria (sin escalar para análisis directo)
lpm_model = LinearRegression()
lpm_model.fit(X_train_cls, y_train_cls)

print('── Coeficientes del Modelo Lineal de Probabilidad (LPM) ──')
for name, coef in zip(features_cls, lpm_model.coef_):
    print(f'  {name}: {coef:.4f}')
print(f'  Intercepto: {lpm_model.intercept_:.4f}')

---
## 🧪 Fase 5 — Evaluación (Evaluation)

Evaluaremos individualmente el rendimiento de los modelos en el conjunto de prueba independiente para verificar si cumplen con las metas de éxito planteadas.

In [ ]:
# ── 1. Evaluación de la Regresión Logística ────────────────────────────────
y_pred_cls = log_model.predict(X_test_cls_scaled)
y_prob_cls = log_model.predict_proba(X_test_cls_scaled)[:, 1]

accuracy = accuracy_score(y_test_cls, y_pred_cls)
print('══════════════════════════════════════════════════════')
print('      EVALUACIÓN CLASIFICACIÓN (REGRESIÓN LOGÍSTICA) ')
print('══════════════════════════════════════════════════════')
print(f'Exactitud (Accuracy): {accuracy * 100:.1f}%')
print('\nReporte de Clasificación:')
print(classification_report(y_test_cls, y_pred_cls, target_names=['No entregó (0)', 'Sí entregó (1)']))

# Gráficas de Evaluación de Clasificación
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de Confusión
cm = confusion_matrix(y_test_cls, y_pred_cls)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No entregó', 'Sí entregó'])
disp.plot(ax=axes[0], cmap='Greens', colorbar=False)
axes[0].set_title('Matriz de Confusión', fontweight='bold')

# Curva ROC y AUC
fpr, tpr, _ = roc_curve(y_test_cls, y_prob_cls)
auc_score = roc_auc_score(y_test_cls, y_prob_cls)
axes[1].plot(fpr, tpr, color='#2980B9', lw=2, label=f'ROC Curve (AUC = {auc_score:.2f})')
axes[1].plot([0, 1], [0, 1], color='#7F8C8D', linestyle='--')
axes[1].set_title('Curva ROC - Regresión Logística', fontweight='bold')
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('eval_clasificacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. Evaluación de la Regresión Lineal ────────────────────────────────────
y_pred_reg = lin_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print('══════════════════════════════════════════════════════')
print('      EVALUACIÓN REGRESIÓN (PREDICCIÓN DE TAREAS)    ')
print('══════════════════════════════════════════════════════')
print(f'Error Absoluto Medio (MAE)  : {mae:.3f}')
print(f'Error Cuadrático Medio (MSE): {mse:.3f}')
print(f'Raíz del MSE (RMSE)         : {rmse:.3f}')
print(f'Coeficiente R² (R-squared)  : {r2:.3f}')

# Gráfico de línea de ajuste de la Regresión Lineal
plt.figure(figsize=(8, 5))
plt.scatter(X_test_reg['Asistencia'], y_test_reg, color='#E67E22', s=80, label='Valores Reales (Prueba)')

# Crear puntos de predicción suaves para la línea de regresión (para asistencia de 50 a 100)
x_dummy = pd.DataFrame({
    'Asistencia': np.linspace(50, 100, 100),
    'Recordatorio_Enviado': np.ones(100) # Asumimos Recordatorio=1 para visualización
})
y_dummy = lin_model.predict(x_dummy)
plt.plot(x_dummy['Asistencia'], y_dummy, color='#2980B9', lw=2, label='Línea de Regresión (Con Recordatorio)')

plt.title('Ajuste de Regresión Lineal: Asistencia vs Tareas Entregadas', fontsize=12, fontweight='bold')
plt.xlabel('% Asistencia')
plt.ylabel('Tareas Anteriores Entregadas')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('eval_regresion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. Comparación Regresión Logística vs LPM ─────────────────────────────────
y_pred_lpm = lpm_model.predict(X_test_cls)

comparativa = pd.DataFrame({
    'Estudiante': df.loc[X_test_cls.index, 'Estudiante'].values,
    'Real': y_test_cls.values,
    'Prob_Logística': y_prob_cls.round(4),
    'Pred_LPM': y_pred_lpm.round(4)
})

print('── Comparativa de Probabilidad Estimada: Regresión Logística vs LPM ──')
print(comparativa.to_string(index=False))
print('\n💡 Nota cómo el Modelo de Probabilidad Lineal (LPM) puede predecir valores')
print('fuera del rango lógico de probabilidades [0, 1] (por ejemplo, mayores que 1.0)')
print('mientras que la Regresión Logística se autolimita entre 0 y 1 mediante la función sigmoide.')

---
## 🚀 Fase 6 — Despliegue y Monitoreo (Deployment & Monitoring)

### **Estrategia de Despliegue**
1. **Interfaz Interactiva**: El modelo ha sido empaquetado en una aplicación web interactiva en **Streamlit** (`app.py`), lo que permite a los docentes interactuar de forma intuitiva con el simulador deslizando variables y obteniendo la probabilidad de riesgo de un estudiante en tiempo real.
2. **Código Abierto y Control de Versiones**: El repositorio completo ha sido estructurado y documentado en **GitHub** para permitir auditoría del código, reproducibilidad y desarrollo colaborativo.

### **Estrategia de Monitoreo y Mantenimiento**
Un modelo predictivo en educación puede perder validez con el tiempo debido a:
- **Deriva de Concepto (Concept Drift)**: Cambios en el comportamiento de los estudiantes por modificaciones curriculares, cambios tecnológicos o alteraciones en los métodos de enseñanza (por ejemplo, transiciones a clases virtuales).
- **Deriva de Datos (Data Drift)**: Variaciones en la distribución de entrada, como variaciones drásticas en la tasa de asistencia general a nivel de escuela.

**Mecanismos sugeridos de monitoreo:**
- Registrar las predicciones hechas y las entregas reales semanales en una base de datos.
- Calcular mensualmente la exactitud (Accuracy) del modelo. Si disminuye por debajo del 85%, activar una alerta para re-entrenar el modelo con los datos más recientes.
- Auditar periódicamente sesgos demográficos o socioeconómicos para asegurar que el modelo no estigmatice a estudiantes vulnerables.